In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install monai 
!pip install nibabel

In [ ]:
import os
import glob
import re
import pandas as pd
import torch
import nibabel as nib
import numpy as np
from monai.transforms import (
    Compose, EnsureChannelFirst, Resize, Orientation, NormalizeIntensity
)
import os
import re
import glob
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import nibabel as nib

from monai.transforms import Compose, Resize, NormalizeIntensity

# =====================================================================
# 1. CONFIGURATION & KAGGLE PATHS
# =====================================================================
# Source is ALREADY processed into .pt files
K_REGIONS = 90
TARGET_SHAPE = (128, 128, 128)
ADNI_SOURCE_DIR = "/kaggle/input"
OASIS_DIR = "/kaggle/input/datasets/ninadaithal/oasis-1-shinohara"

OUTPUT_DIR = "/kaggle/working/model_ready_data"
os.makedirs(os.path.join(OUTPUT_DIR, "source_adni"), exist_ok=True)

os.makedirs(os.path.join(OUTPUT_DIR, "source_adni"), exist_ok=True)
for label in ["CN", "MCI", "AD"]:
    os.makedirs(os.path.join(OUTPUT_DIR, "target_oasis", label), exist_ok=True)

# -------------------------------------------------
# 2) RESOLUCIÓN ROBUSTA DE RAÍCES DE DATASET
# -------------------------------------------------
def resolve_dataset_root(slug_hint: str) -> str:
    candidates = sorted(glob.glob(f"/kaggle/input/*{slug_hint}*"))
    if not candidates:
        raise FileNotFoundError(f"No se encontró un dataset con patrón: {slug_hint}")
    return candidates[0]

# ADNI_SOURCE_DIR = resolve_dataset_root("adnisanju")
# OASIS_DIR = resolve_dataset_root("oasis-1-shinohara")

# -------------------------------------------------
# 3) TRANSFORMACIONES MRI
# Nota: no usamos MONAI Orientation aquí, porque
#       al hacer get_fdata() ya se pierde el affine.
#       La orientación se corrige con nibabel antes.
# -------------------------------------------------
mri_transforms = Compose([
    Resize(spatial_size=TARGET_SHAPE),
    NormalizeIntensity(nonzero=True, channel_wise=True),
])

OASIS_ID_RE = re.compile(r"(OAS1_\d{4})")

def extract_oasis_base_id(text: str):
    m = OASIS_ID_RE.search(str(text))
    return m.group(1) if m else None

def load_mri_tensor(filepath: str) -> torch.Tensor:
    img = nib.load(filepath)
    img = nib.as_closest_canonical(img)  # conserva la geometría antes de extraer el array
    vol = img.get_fdata(dtype=np.float32)

    if vol.ndim == 4:
        vol = vol[..., 0]

    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)
    x = torch.from_numpy(vol).unsqueeze(0)  # (1, H, W, D)
    x = mri_transforms(x)                   # (1, 128, 128, 128)
    return x.to(torch.float32)

# -------------------------------------------------
# 4) INVENTARIO SOURCE (ADNI)
# Importante:
#   - No generar c_target ni g_bar aleatorios.
#   - Solo inventariar y validar.
# -------------------------------------------------
def try_extract_tensor_shape(obj):
    if torch.is_tensor(obj):
        return tuple(obj.shape)
    if isinstance(obj, dict):
        for key in ["image", "x", "tensor", "mri", "volume"]:
            if key in obj and torch.is_tensor(obj[key]):
                return tuple(obj[key].shape)
    return None

def inventory_source_domain():
    print("--- Inventorying source domain (ADNI) ---")
    search_pattern = os.path.join(ADNI_SOURCE_DIR, "**", "*.pt")
    all_pt_files = glob.glob(search_pattern, recursive=True)

    source_metadata = []

    for file_path in sorted(all_pt_files):
        path_upper = file_path.upper()

        if "/MCI/" in path_upper or "\\MCI\\" in path_upper:
            label = "MCI"
        elif "/AD/" in path_upper or "\\AD\\" in path_upper:
            label = "AD"
        elif "/CN/" in path_upper or "\\CN\\" in path_upper:
            label = "CN"
        else:
            continue

        sub_id = os.path.splitext(os.path.basename(file_path))[0]

        tensor_shape = None
        try:
            obj = torch.load(file_path, map_location="cpu")
            tensor_shape = try_extract_tensor_shape(obj)
        except Exception:
            tensor_shape = None

        source_metadata.append({
            "Subject_ID": sub_id,
            "Label": label,
            "Raw_File_Path": file_path,
            "Tensor_Shape": tensor_shape,
            "Concept_Target_Path": None,   # pendiente real
            "Jacobian_Path": None,         # pendiente real
        })

    df_source = pd.DataFrame(source_metadata)
    df_source.to_csv(os.path.join(OUTPUT_DIR, "source_labels.csv"), index=False)

    print(f"[OK] Source subjects inventoried: {len(df_source)}")
    if len(df_source) > 0:
        print(df_source["Label"].value_counts().to_string())

# -------------------------------------------------
# 5) ETIQUETAS OASIS
# -------------------------------------------------
def get_oasis_label_dict():
    print("--- Parsing OASIS clinical CSV ---")
    csv_files = glob.glob(os.path.join(OASIS_DIR, "**", "*oasis_cross-sectional*.csv"), recursive=True)
    if not csv_files:
        raise FileNotFoundError("No se encontró oasis_cross-sectional.csv")

    df = pd.read_csv(csv_files[0])

    label_dict = {}
    for _, row in df.iterrows():
        base_id = extract_oasis_base_id(row.get("ID", ""))
        if base_id is None:
            continue

        cdr = row.get("CDR", np.nan)

        if pd.isna(cdr):
            label = "Exclude"
        elif float(cdr) == 0.0:
            label = "CN"
        elif float(cdr) == 0.5:
            label = "MCI"   # convención experimental
        else:
            label = "AD"

        label_dict[base_id] = label

    return label_dict

# -------------------------------------------------
# 6) TARGET DOMAIN (OASIS)
# Importante:
#   - un solo volumen por sujeto
#   - evitar sobrescritura
# -------------------------------------------------
def process_target_domain():
    print("--- Processing target domain (OASIS) ---")
    label_dict = get_oasis_label_dict()

    target_files = sorted(glob.glob(os.path.join(OASIS_DIR, "**", "*.nii*"), recursive=True))
    if not target_files:
        raise FileNotFoundError("No se encontraron archivos NIfTI en OASIS")

    # agrupar por sujeto para evitar sobrescritura
    subject_to_files = {}
    for f in target_files:
        base_id = extract_oasis_base_id(f)
        if base_id is None:
            continue
        if label_dict.get(base_id, "Exclude") == "Exclude":
            continue
        subject_to_files.setdefault(base_id, []).append(f)

    target_metadata = []

    for base_id, files in sorted(subject_to_files.items()):
        label = label_dict[base_id]

        # selección determinista de un volumen por sujeto
        chosen_file = sorted(files)[0]

        try:
            x = load_mri_tensor(chosen_file)
            save_path = os.path.join(OUTPUT_DIR, "target_oasis", label, f"{base_id}_MRI.pt")
            torch.save(x, save_path)

            target_metadata.append({
                "Subject_ID": base_id,
                "Label": label,
                "Raw_File_Path": chosen_file,
                "Processed_File_Path": save_path,
                "N_scans_found": len(files),
            })

        except Exception as e:
            print(f"[ERROR] {base_id}: {e}")

    df_target = pd.DataFrame(target_metadata)
    df_target.to_csv(os.path.join(OUTPUT_DIR, "target_labels.csv"), index=False)

    print(f"[OK] Target subjects processed: {len(df_target)}")
    if len(df_target) > 0:
        print(df_target["Label"].value_counts().to_string())


# ==============================================================================
# 1) CONFIGURATION
# ==============================================================================

TARGET_SHAPE = (128, 128, 128)
CLASS_NAMES = ("CN", "MCI", "AD")
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

# Kaggle input roots. The resolver below is robust to names such as:
# /kaggle/input/adnidataset
# /kaggle/input/adni-dataset
# /kaggle/input/datasets/sanjukaggling/adnidataset
KAGGLE_INPUT = Path("/kaggle/input")

ADNI_SLUG_HINTS = ("adnidataset", "adni")
OASIS_SLUG_HINTS = ("oasis-1-shinohara", "oasis")

OUTPUT_DIR = Path("/kaggle/working/model_ready_data")
ADNI_OUT_DIR = OUTPUT_DIR / "source_adni"
OASIS_OUT_DIR = OUTPUT_DIR / "target_oasis"

SUPPORTED_EXTENSIONS = (
    ".pt", ".pth",
    ".nii", ".nii.gz", ".img", ".hdr", ".mgz", ".mgh",
    ".npy", ".npz",
)

ALLOW_UNSAFE_TORCH_LOAD = False
KEEP_ONE_SCAN_PER_SUBJECT_ADNI = False

for domain_dir in (ADNI_OUT_DIR, OASIS_OUT_DIR):
    for label in CLASS_NAMES:
        (domain_dir / label).mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "class_to_idx.json", "w") as f:
    json.dump(CLASS_TO_IDX, f, indent=2)

mri_transforms = Compose([
    Resize(spatial_size=TARGET_SHAPE, mode="trilinear"),
    NormalizeIntensity(nonzero=True, channel_wise=True),
])

# # =====================================================================
# # 5. EXECUTE PIPELINE
# # =====================================================================
# if __name__ == "__main__":
#     inventory_source_domain()
process_target_domain()
print("\n[PIPELINE COMPLETE] Data is formatted as .pt tensors, categorized in folders, and mapped in CSVs.")

In [ ]:



# ==============================================================================
# 2) GENERAL UTILITIES
# ==============================================================================

def lower_name(path):
    return str(path).lower()

def is_supported_file(path):
    name = lower_name(path)
    return any(name.endswith(ext) for ext in SUPPORTED_EXTENSIONS)

def strip_medical_suffix(path):
    name = Path(path).name
    for suffix in [".nii.gz", ".nii", ".img", ".hdr", ".mgz", ".mgh", ".pt", ".pth", ".npy", ".npz"]:
        if name.lower().endswith(suffix):
            return name[:-len(suffix)]
    return Path(path).stem

def sanitize_id(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text if text else "unknown"

def make_unique_path(path):
    path = Path(path)
    if not path.exists():
        return path
    stem, suffix = path.stem, path.suffix
    parent = path.parent
    k = 1
    while True:
        candidate = parent / f"{stem}_{k:03d}{suffix}"
        if not candidate.exists():
            return candidate
        k += 1

def resolve_dataset_root(slug_hints, require_class_dirs=False):
    """
    Finds the most plausible dataset root in /kaggle/input.
    If require_class_dirs=True, it prioritizes roots containing CN/MCI/AD directories.
    """
    candidates = []

    if KAGGLE_INPUT.exists():
        candidates.append(KAGGLE_INPUT)

        for p in KAGGLE_INPUT.iterdir():
            if p.is_dir():
                candidates.append(p)

        for hint in slug_hints:
            candidates.extend([Path(p) for p in glob.glob(str(KAGGLE_INPUT / f"*{hint}*"))])
            candidates.extend([Path(p) for p in glob.glob(str(KAGGLE_INPUT / "**" / f"*{hint}*"), recursive=True)])

    # Remove duplicated paths while preserving order.
    seen = set()
    candidates = [p for p in candidates if not (str(p) in seen or seen.add(str(p)))]

    def score_root(root):
        score = 0
        root_l = str(root).lower()
        for hint in slug_hints:
            if hint.lower() in root_l:
                score += 100

        if require_class_dirs:
            class_dirs = find_class_dirs(root, silent=True)
            score += 20 * len(class_dirs)
            # Prefer the shallowest root that contains the three class dirs.
            score -= len(root.parts)

        return score

    if not candidates:
        raise FileNotFoundError("No dataset roots were found under /kaggle/input.")

    ranked = sorted(candidates, key=score_root, reverse=True)

    if require_class_dirs:
        for root in ranked:
            if len(find_class_dirs(root, silent=True)) >= 1:
                return root

    return ranked[0]

def find_class_dirs(root, silent=False):
    """
    Searches recursively for directories named exactly CN, MCI, and AD.
    Returns one directory per class, choosing the shallowest candidate.
    """
    root = Path(root)
    matches = {label: [] for label in CLASS_NAMES}

    if not root.exists():
        if not silent:
            print(f"[WARN] Root does not exist: {root}")
        return {}

    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            d_upper = d.upper()
            if d_upper in matches:
                matches[d_upper].append(Path(dirpath) / d)

    class_dirs = {}
    for label, dirs in matches.items():
        if dirs:
            class_dirs[label] = sorted(dirs, key=lambda p: (len(p.parts), str(p)))[0]

    return class_dirs

def print_tree_hint(root, max_files=50):
    print(f"\n--- First files under {root} ---")
    counter = 0
    for dirpath, _, filenames in os.walk(root):
        for filename in filenames:
            print(Path(dirpath) / filename)
            counter += 1
            if counter >= max_files:
                return

# ==============================================================================
# 3) LOADING AND PREPROCESSING MRI VOLUMES
# ==============================================================================

def safe_torch_load(path):
    """
    Loads .pt/.pth files. By default, it avoids unsafe pickle loading.
    If your Kaggle .pt files are trusted and require pickle objects,
    set ALLOW_UNSAFE_TORCH_LOAD=True.
    """
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")
    except Exception as e:
        if not ALLOW_UNSAFE_TORCH_LOAD:
            raise RuntimeError(
                f"Could not safely load {path}. If this file is trusted, "
                "set ALLOW_UNSAFE_TORCH_LOAD=True."
            ) from e
        return torch.load(path, map_location="cpu", weights_only=False)

def extract_tensor_from_object(obj):
    """
    Extracts the MRI array/tensor from common .pt containers.
    Supports raw Tensor/ndarray, dicts, and tuples/lists.
    """
    if torch.is_tensor(obj) or isinstance(obj, np.ndarray):
        return obj

    if isinstance(obj, dict):
        preferred_keys = [
            "image", "img", "x", "X", "tensor", "mri", "MRI",
            "volume", "vol", "data", "scan", "arr", "array"
        ]
        for key in preferred_keys:
            if key in obj:
                try:
                    return extract_tensor_from_object(obj[key])
                except Exception:
                    pass

        for value in obj.values():
            try:
                candidate = extract_tensor_from_object(value)
                if candidate is not None:
                    return candidate
            except Exception:
                continue

    if isinstance(obj, (list, tuple)):
        for value in obj:
            try:
                candidate = extract_tensor_from_object(value)
                if candidate is not None:
                    return candidate
            except Exception:
                continue

    raise ValueError("No tensor-like MRI volume was found inside the loaded object.")

def to_channel_first_3d(x):
    """
    Converts a 3D/4D MRI array into shape (1, H, W, D).
    For 4D MRI volumes, it keeps the first channel/timepoint when needed.
    """
    if isinstance(x, np.ndarray):
        x = torch.from_numpy(x)

    if not torch.is_tensor(x):
        x = torch.as_tensor(x)

    x = x.detach().cpu().float()
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    # Remove irrelevant singleton dimensions, but preserve a possible channel axis.
    while x.ndim > 4 and 1 in x.shape:
        singleton_axis = list(x.shape).index(1)
        x = x.squeeze(singleton_axis)

    if x.ndim == 5:
        # Common case: (B, C, H, W, D) or (B, H, W, D, C), keep first batch.
        if x.shape[0] == 1:
            x = x[0]
        else:
            x = x[0]

    if x.ndim == 4:
        # Channel-first: (C, H, W, D)
        if x.shape[0] <= 10:
            x = x[:1]
        # Channel-last: (H, W, D, C)
        elif x.shape[-1] <= 10:
            x = x.permute(3, 0, 1, 2)[:1]
        # 4D acquisition without explicit channel convention: keep first volume.
        else:
            x = x[..., 0].unsqueeze(0)

    elif x.ndim == 3:
        x = x.unsqueeze(0)

    else:
        raise ValueError(f"Expected a 3D/4D MRI volume, but got shape {tuple(x.shape)}.")

    return x.contiguous()

def load_nifti_tensor(path):
    img = nib.load(str(path))
    img = nib.as_closest_canonical(img)
    vol = img.get_fdata(dtype=np.float32)
    return to_channel_first_3d(vol)

def load_numpy_tensor(path):
    path = Path(path)
    if path.name.lower().endswith(".npy"):
        arr = np.load(path)
        return to_channel_first_3d(arr)

    data = np.load(path)
    if isinstance(data, np.lib.npyio.NpzFile):
        keys = list(data.keys())
        keys = sorted(keys, key=lambda k: 0 if np.asarray(data[k]).ndim >= 3 else 1)
        return to_channel_first_3d(data[keys[0]])

    return to_channel_first_3d(data)

def load_pt_tensor(path):
    obj = safe_torch_load(path)
    x = extract_tensor_from_object(obj)
    return to_channel_first_3d(x)

import os
from pathlib import Path
import numpy as np
import torch
import SimpleITK as sitk


def lower_name(path):
    return str(path).lower()


def numpy_to_torch_safe(arr, dtype=np.float32):
    arr = np.asarray(arr, dtype=dtype)
    if any(s < 0 for s in arr.strides):
        arr = arr.copy()
    arr = np.ascontiguousarray(arr)
    return torch.from_numpy(arr)


def find_dicom_series_dir(path: Path) -> Path:
    """
    Si `path` es un archivo .dcm, devuelve su carpeta.
    Si `path` es una carpeta, intenta encontrar dentro la carpeta más profunda
    que realmente contiene una serie DICOM válida.
    """
    if path.is_file():
        return path.parent

    if not path.is_dir():
        raise FileNotFoundError(f"No existe la ruta: {path}")

    # buscar de abajo hacia arriba una carpeta con archivos DICOM
    candidate_dirs = []
    for root, _, files in os.walk(path):
        dcm_files = [f for f in files if f.lower().endswith(".dcm")]
        if len(dcm_files) > 0:
            candidate_dirs.append(Path(root))

    if len(candidate_dirs) == 0:
        raise FileNotFoundError(f"No se encontraron archivos DICOM dentro de: {path}")

    # preferir carpetas más profundas
    candidate_dirs = sorted(candidate_dirs, key=lambda p: len(str(p).split(os.sep)), reverse=True)

    for cdir in candidate_dirs:
        try:
            series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(cdir))
            if series_ids is not None and len(series_ids) > 0:
                return cdir
        except Exception:
            continue

    raise RuntimeError(f"No se pudo encontrar una serie DICOM válida dentro de: {path}")


def load_dicom_series_tensor(path: Path) -> torch.Tensor:
    """
    Carga una serie DICOM completa y devuelve tensor con shape [1, H, W, D].
    """
    series_dir = find_dicom_series_dir(path)

    series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(series_dir))
    if not series_ids:
        raise RuntimeError(f"No se encontraron series DICOM en: {series_dir}")

    # si hay varias series en la misma carpeta, elegir la de mayor número de slices
    best_files = None
    best_len = -1

    for sid in series_ids:
        files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(series_dir), sid)
        if len(files) > best_len:
            best_len = len(files)
            best_files = files

    if best_files is None or len(best_files) == 0:
        raise RuntimeError(f"No se pudieron resolver los archivos DICOM en: {series_dir}")

    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(best_files)
    image = reader.Execute()

    # SimpleITK -> numpy: [D, H, W]
    arr = sitk.GetArrayFromImage(image).astype(np.float32)

    # convertir a [H, W, D]
    arr = np.transpose(arr, (1, 2, 0))

    # tensor final [1, H, W, D]
    x = numpy_to_torch_safe(arr).unsqueeze(0)
    return x


def load_mri_tensor(path):
    path = Path(path)

    if path.is_dir():
        # tratar como carpeta DICOM
        x = load_dicom_series_tensor(path)

    else:
        name = lower_name(path)

        if name.endswith((".nii", ".nii.gz", ".img", ".hdr", ".mgz", ".mgh")):
            x = load_nifti_tensor(path)
        elif name.endswith((".pt", ".pth")):
            x = load_pt_tensor(path)
        elif name.endswith((".npy", ".npz")):
            x = load_numpy_tensor(path)
        elif name.endswith(".dcm"):
            x = load_dicom_series_tensor(path)
        else:
            raise ValueError(f"Unsupported file extension: {path}")

    x = mri_transforms(x)
    return x.to(torch.float32).contiguous()

# ==============================================================================
# 4) ADNI PREPROCESSING: CN / MCI / AD DIRECTORIES
# ==============================================================================

import os
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import SimpleITK as sitk


def robust_intensity_normalization(x: torch.Tensor, pmin=1.0, pmax=99.0):
    """
    x: [1,H,W,D]
    """
    vals = x[x > 0]
    if vals.numel() == 0:
        return x

    lo = torch.quantile(vals, pmin / 100.0)
    hi = torch.quantile(vals, pmax / 100.0)

    x = torch.clamp(x, lo, hi)
    mean = vals.mean()
    std = vals.std().clamp_min(1e-6)
    x = (x - mean) / std
    return x


def resize_3d_tensor(x: torch.Tensor, target_shape=(128, 128, 128)):
    """
    x: [1,H,W,D] -> devuelve [1,Ht,Wt,Dt]
    """
    x5 = x.unsqueeze(0)                      # [1,1,H,W,D]
    x5 = x5.permute(0, 1, 4, 2, 3)          # [1,1,D,H,W] para interpolate 3D
    x5 = F.interpolate(
        x5,
        size=(target_shape[2], target_shape[0], target_shape[1]),
        mode="trilinear",
        align_corners=False,
    )
    x5 = x5.permute(0, 1, 3, 4, 2)          # [1,1,H,W,D]
    return x5.squeeze(0)                    # [1,H,W,D]


def center_crop_or_pad_3d(x: torch.Tensor, target_shape=(160, 192, 160)):
    """
    x: [1,H,W,D]
    """
    _, H, W, D = x.shape
    tH, tW, tD = target_shape

    out = torch.zeros((1, tH, tW, tD), dtype=x.dtype)

    h0_src = max((H - tH) // 2, 0)
    w0_src = max((W - tW) // 2, 0)
    d0_src = max((D - tD) // 2, 0)

    h1_src = min(h0_src + tH, H)
    w1_src = min(w0_src + tW, W)
    d1_src = min(d0_src + tD, D)

    crop = x[:, h0_src:h1_src, w0_src:w1_src, d0_src:d1_src]

    _, cH, cW, cD = crop.shape
    h0_dst = max((tH - cH) // 2, 0)
    w0_dst = max((tW - cW) // 2, 0)
    d0_dst = max((tD - cD) // 2, 0)

    out[:, h0_dst:h0_dst+cH, w0_dst:w0_dst+cW, d0_dst:d0_dst+cD] = crop
    return out


def mri_transforms(x: torch.Tensor, target_shape=(128, 128, 128)):
    """
    Preprocesamiento compatible con pipeline 3D tipo OASIS-Shinohara.
    Espera x con shape [1,H,W,D].
    """
    x = x.to(torch.float32)

    # corregir NaN / inf
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    # crop/pad suave antes de resize
    x = center_crop_or_pad_3d(x, target_shape=(160, 192, 160))

    # resize final
    x = resize_3d_tensor(x, target_shape=target_shape)

    # normalización robusta
    x = robust_intensity_normalization(x)

    return x.contiguous()



ADNI_SUBJECT_RE = re.compile(r"(\d{3}_S_\d{4})", re.IGNORECASE)

def extract_adni_subject_id(path):
    text = str(path)
    m = ADNI_SUBJECT_RE.search(text)
    if m:
        return sanitize_id(m.group(1))
    return sanitize_id(strip_medical_suffix(path))

def iter_adni_files_by_label(adni_root):
    class_dirs = find_class_dirs(adni_root)
    if not class_dirs:
        raise FileNotFoundError(
            f"No CN/MCI/AD directories were found under {adni_root}. "
            "Check that the Kaggle ADNI dataset was added to the notebook."
        )

    print("--- ADNI class directories detected ---")
    for label, class_dir in class_dirs.items():
        print(f"{label}: {class_dir}")

    for label, class_dir in class_dirs.items():
        for dirpath, _, filenames in os.walk(class_dir):
            for filename in filenames:
                path = Path(dirpath) / filename
                if is_supported_file(path):
                    yield label, path

def find_adni_mprage_series_dirs(adni_root: str):
    """
    Busca directorios que contengan una serie DICOM MP-RAGE / MPRAGE.
    """
    adni_root = Path(adni_root)
    candidates = []

    for root, dirs, files in os.walk(adni_root):
        root_path = Path(root)
        root_low = str(root_path).lower()

        # filtrar por nombre de secuencia
        if ("mp-rage" in root_low) or ("mprage" in root_low):
            dcm_files = [f for f in files if f.lower().endswith(".dcm")]
            if len(dcm_files) > 0:
                candidates.append(root_path)

    return sorted(set(candidates))

def preprocess_adni_to_pt(
    adni_root: str,
    labels_csv: str,
    out_root: str,
    target_shape=(128, 128, 128),
):
    """
    labels_csv debe tener columnas:
        Subject_ID, Label
    donde Subject_ID coincide con el sujeto ADNI, por ejemplo 002_S_0619
    """
    adni_root = Path(adni_root)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(labels_csv)

    saved_rows = []
    errors = []

    for _, row in df.iterrows():
        subject_id = str(row["Subject"])
        label = str(row["Group"])

        try:
            # buscar la carpeta del sujeto
            subject_dirs = list(adni_root.rglob(subject_id))
            subject_dirs = [p for p in subject_dirs if p.is_dir()]

            if len(subject_dirs) == 0:
                raise FileNotFoundError(f"No se encontró carpeta para sujeto {subject_id}")

            # preferir una carpeta que contenga MP-RAGE
            chosen_series = None
            for subj_dir in subject_dirs:
                for series_dir in find_adni_mprage_series_dirs(str(subj_dir)):
                    chosen_series = series_dir
                    break
                if chosen_series is not None:
                    break

            if chosen_series is None:
                raise FileNotFoundError(f"No se encontró serie MP-RAGE para {subject_id}")

            x = load_dicom_series_tensor(chosen_series)
            x = mri_transforms(x, target_shape=target_shape)

            label_dir = out_root / label
            label_dir.mkdir(parents=True, exist_ok=True)

            out_path = label_dir / f"{subject_id}_MRI.pt"
            torch.save(x.cpu(), out_path)

            saved_rows.append({
                "Subject_ID": subject_id,
                "Label": label,
                "Processed_File_Path": str(out_path),
            })

            print(f"[OK] {label} | {subject_id} -> {out_path}")

        except Exception as e:
            errors.append({
                "Subject_ID": subject_id,
                "Label": label,
                "error": str(e),
            })
            print(f"[ERROR] {label} | {subject_id}: {e}")

    df_saved = pd.DataFrame(saved_rows)
    df_errors = pd.DataFrame(errors)

    df_saved.to_csv(out_root / "adni_preprocessed_index.csv", index=False)
    df_errors.to_csv(out_root / "adni_preprocess_errors.csv", index=False)

    return df_saved, df_errors

# def process_adni_domain(adni_root=None):
#     print("\n==============================")
#     print("Processing source domain: ADNI")
#     print("==============================")

#     if adni_root is None:
#         adni_root = resolve_dataset_root(ADNI_SLUG_HINTS, require_class_dirs=True)
#     adni_root = Path(adni_root)

#     records = []
#     processed_subjects = set()

#     for label, raw_path in sorted(iter_adni_files_by_label(adni_root), key=lambda z: str(z[1])):
#         subject_id = extract_adni_subject_id(raw_path)

#         if KEEP_ONE_SCAN_PER_SUBJECT_ADNI and subject_id in processed_subjects:
#             continue

#         try:
#             x = load_mri_tensor(raw_path)

#             save_name = f"{subject_id}_MRI.pt"
#             save_path = make_unique_path(ADNI_OUT_DIR / label / save_name)
#             torch.save(x, save_path)

#             processed_subjects.add(subject_id)

#             records.append({
#                 "Subject_ID": subject_id,
#                 "Label": label,
#                 "Label_Index": CLASS_TO_IDX[label],
#                 "Raw_File_Path": str(raw_path),
#                 "Processed_File_Path": str(save_path),
#                 "Tensor_Shape": tuple(x.shape),
#             })

#         except Exception as e:
#             records.append({
#                 "Subject_ID": subject_id,
#                 "Label": label,
#                 "Label_Index": CLASS_TO_IDX[label],
#                 "Raw_File_Path": str(raw_path),
#                 "Processed_File_Path": None,
#                 "Tensor_Shape": None,
#                 "Error": repr(e),
#             })
#             print(f"[ERROR] ADNI {label} | {raw_path}: {e}")

#     df = pd.DataFrame(records)
#     df.to_csv(OUTPUT_DIR / "source_adni_labels.csv", index=False)

#     ok_df = df[df["Processed_File_Path"].notna()] if len(df) else df
#     print(f"\n[OK] ADNI files processed: {len(ok_df)} / {len(df)}")
#     if len(ok_df):
#         print(ok_df["Label"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string())

#     return df

# ==============================================================================
# 5) OASIS PREPROCESSING: FOLLOWING CDR-BASED LABELS
# ==============================================================================

OASIS_ID_RE = re.compile(r"(OAS1_\d{4})", re.IGNORECASE)

def extract_oasis_base_id(text):
    m = OASIS_ID_RE.search(str(text))
    return m.group(1).upper() if m else None

def find_oasis_clinical_csv(oasis_root):
    oasis_root = Path(oasis_root)

    candidates = list(oasis_root.rglob("*oasis_cross-sectional*.csv"))
    if not candidates:
        candidates = list(oasis_root.rglob("*.csv"))

    for csv_path in candidates:
        try:
            df_head = pd.read_csv(csv_path, nrows=5)
            cols = {c.lower(): c for c in df_head.columns}
            has_cdr = "cdr" in cols
            has_id = any(k in cols for k in ["id", "subject id", "subject_id", "subject"])
            if has_cdr and has_id:
                return csv_path
        except Exception:
            continue

    raise FileNotFoundError(
        f"No OASIS clinical CSV with ID and CDR columns was found under {oasis_root}."
    )

# def get_oasis_label_dict(oasis_root):
#     csv_path = find_oasis_clinical_csv(oasis_root)
#     print(f"Clinical OASIS CSV: {csv_path}")

#     df = pd.read_csv(csv_path)
#     id_col = None
#     for candidate in ["ID", "Subject ID", "Subject_ID", "Subject"]:
#         if candidate in df.columns:
#             id_col = candidate
#             break
#     if id_col is None:
#         id_col = df.columns[0]

#     if "CDR" not in df.columns:
#         raise ValueError("The OASIS clinical CSV does not contain a CDR column.")

#     label_dict = {}

#     for _, row in df.iterrows():
#         base_id = extract_oasis_base_id(row.get(id_col, ""))
#         if base_id is None:
#             continue

#         cdr = row.get("CDR", np.nan)
#         if pd.isna(cdr):
#             continue

#         cdr = float(cdr)

#         if cdr == 0.0:
#             label = "CN"
#         elif cdr == 0.5:
#             label = "MCI"
#         else:
#             label = "AD"

#         label_dict[base_id] = label

#     return label_dict

def choose_oasis_scan(files):
    """
    Deterministic scan selection. It prioritizes processed anatomical files if names contain
    common markers; otherwise, it uses alphabetical order.
    """
    def score(path):
        name = lower_name(path)
        s = 0
        for token in ["mpr", "t1", "brain", "struc", "processed", "masked", "talairach"]:
            if token in name:
                s -= 1
        return (s, str(path))

    return sorted(files, key=score)[0]

def process_oasis_domain(oasis_root=None):
    print("\n===============================")
    print("Processing target domain: OASIS")
    print("===============================")

    if oasis_root is None:
        oasis_root = resolve_dataset_root(OASIS_SLUG_HINTS, require_class_dirs=False)
    oasis_root = Path(oasis_root)

    label_dict = get_oasis_label_dict(oasis_root)

    files = [
        p for p in oasis_root.rglob("*")
        if p.is_file()
        and is_supported_file(p)
        and lower_name(p).endswith((".nii", ".nii.gz", ".img", ".mgz", ".mgh", ".npy", ".npz", ".pt", ".pth"))
    ]

    subject_to_files = {}
    for path in files:
        base_id = extract_oasis_base_id(path)
        if base_id is None:
            continue
        if base_id not in label_dict:
            continue
        subject_to_files.setdefault(base_id, []).append(path)

    records = []

    for base_id, subject_files in sorted(subject_to_files.items()):
        label = label_dict[base_id]
        raw_path = choose_oasis_scan(subject_files)

        try:
            x = load_mri_tensor(raw_path)

            save_path = make_unique_path(OASIS_OUT_DIR / label / f"{base_id}_MRI.pt")
            torch.save(x, save_path)

            records.append({
                "Subject_ID": base_id,
                "Label": label,
                "Label_Index": CLASS_TO_IDX[label],
                "Raw_File_Path": str(raw_path),
                "Processed_File_Path": str(save_path),
                "Tensor_Shape": tuple(x.shape),
                "N_scans_found": len(subject_files),
            })

        except Exception as e:
            records.append({
                "Subject_ID": base_id,
                "Label": label,
                "Label_Index": CLASS_TO_IDX[label],
                "Raw_File_Path": str(raw_path),
                "Processed_File_Path": None,
                "Tensor_Shape": None,
                "N_scans_found": len(subject_files),
                "Error": repr(e),
            })
            print(f"[ERROR] OASIS {label} | {base_id}: {e}")

    df = pd.DataFrame(records)
    df.to_csv(OUTPUT_DIR / "target_oasis_labels.csv", index=False)

    ok_df = df[df["Processed_File_Path"].notna()] if len(df) else df
    print(f"\n[OK] OASIS subjects processed: {len(ok_df)} / {len(df)}")
    if len(ok_df):
        print(ok_df["Label"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string())

    return df

# ==============================================================================
# 6) EXECUTION
# ==============================================================================

# if __name__ == "__main__":
    # Optional: inspect /kaggle/input if path resolution fails.
    # print_tree_hint(KAGGLE_INPUT, max_files=100)

# adni_df = process_adni_domain(adni_root= "/kaggle/input/datasets/sanjukaggling/adnidataset/ADNI_dataset")

df_saved, df_errors = preprocess_adni_to_pt(
    adni_root= "/kaggle/input/datasets/sanjukaggling/adnidataset/ADNI_dataset",
    labels_csv= "/kaggle/input/datasets/sanjukaggling/adnidataset/ADNI_dataset/ad_new_2_19_2026.csv",
    out_root= ADNI_OUT_DIR)
# oasis_df = process_oasis_domain(oasis_root= "/kaggle/input/datasets/ninadaithal/oasis-1-shinohara")
# oasis_df =process_target_domain()
print("\n[PIPELINE COMPLETE] Data is formatted as .pt tensors, categorized in folders, and mapped in CSVs.")
print("\n[PIPELINE COMPLETE]")
print(f"ADNI tensors:  {ADNI_OUT_DIR}")
print(f"OASIS tensors: {OASIS_OUT_DIR}")
print(f"Metadata:      {OUTPUT_DIR}")

In [ ]:
df_saved, df_errors

In [ ]:
# oasis_df